# Lesson 3 — Error Mitigation Techniques

**Quantum Computing with Qiskit II**

This notebook accompanies Lesson 3. Work through the cells in order.

**Topics covered:**
- Measurement error mitigation via calibration matrix inversion
- Zero-noise extrapolation with circuit-level gate folding
- Richardson extrapolation (linear and quadratic)
- Qiskit Runtime `EstimatorV2` with resilience levels 0, 1, and 2

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime --quiet

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit:             ", qiskit.__version__)
print("Qiskit Aer:         ", qiskit_aer.__version__)
print("Qiskit IBM Runtime: ", qiskit_ibm_runtime.__version__)

In [ ]:
# Core imports used throughout the notebook
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

print("Imports complete.")

## 1. Measurement Error Mitigation

Readout errors occur at the final measurement step, independent of gate errors.
We characterize them by building a **calibration matrix** $M$ where $M[i][j]$ is the
probability of measuring outcome $i$ when state $j$ was prepared.

The noisy probability vector satisfies $\mathbf{p}_\mathrm{noisy} = M\,\mathbf{p}_\mathrm{ideal}$,
so the corrected estimate is $\mathbf{p}_\mathrm{corrected} = M^{-1}\,\mathbf{p}_\mathrm{noisy}$.

In [ ]:
# Noise model: 6% symmetric readout error, no gate errors
# ReadoutError([[P(measure j | prep 0)], [P(measure j | prep 1)]])
ro_err = ReadoutError([[0.94, 0.06],
                       [0.06, 0.94]])

nm_ro = NoiseModel()
nm_ro.add_all_qubit_readout_error(ro_err)
backend_ro = AerSimulator(noise_model=nm_ro)

shots_cal = 16384

# ------------------------------------------------------------------ #
# Calibration: prepare each 2-qubit basis state and measure.
# state_labels: rightmost character = qubit 0 (Qiskit convention).
# ------------------------------------------------------------------ #
state_labels = ['00', '01', '10', '11']
M_cal = np.zeros((4, 4))

for j, prep_label in enumerate(state_labels):
    qc = QuantumCircuit(2, 2)
    for qubit_idx, bit_char in enumerate(reversed(prep_label)):
        if bit_char == '1':
            qc.x(qubit_idx)
    qc.measure([0, 1], [0, 1])
    counts = backend_ro.run(transpile(qc, backend_ro), shots=shots_cal).result().get_counts()
    for i, meas_label in enumerate(state_labels):
        M_cal[i, j] = counts.get(meas_label, 0) / shots_cal

print("Calibration matrix (rows: measured, cols: prepared):")
print(np.round(M_cal, 4))
# Expected: diagonal ~0.8836 (= 0.94^2), near-zero corners ~0.0036 (= 0.06^2),
# and off-diagonal entries ~0.0564 (= 0.94 * 0.06) for single-qubit errors.

In [ ]:
# Run the Bell state circuit under readout noise and apply calibration correction
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure([0, 1], [0, 1])

counts_ro = backend_ro.run(transpile(qc_bell, backend_ro), shots=shots_cal).result().get_counts()

# Convert to probability vector (same order as state_labels)
p_noisy = np.array([counts_ro.get(s, 0) / shots_cal for s in state_labels])

# Apply M^{-1}
M_inv        = np.linalg.inv(M_cal)
p_corrected  = M_inv @ p_noisy
# Clip small negatives from shot noise in calibration and renormalize
p_corrected  = np.clip(p_corrected, 0, 1)
p_corrected /= p_corrected.sum()

print(f"{'State':>6}  {'Ideal':>8}  {'Noisy':>8}  {'Corrected':>10}")
print("-" * 38)
for s, p_n, p_c in zip(state_labels, p_noisy, p_corrected):
    ideal = 0.5 if s in ('00', '11') else 0.0
    print(f"  |{s}>  {ideal:>8.4f}  {p_n:>8.4f}  {p_c:>10.4f}")

# Expected: noisy shows ~0.44 for |00> and |11>, ~0.056 for |01> and |10>.
# After correction: |00> and |11> recover to ~0.500, error states collapse to ~0.

In [ ]:
# Visualize: before and after correction
x = np.arange(4)
width = 0.3

ideal_probs = [0.5, 0.0, 0.0, 0.5]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(x - width, ideal_probs,  width, label='Ideal',     color='black',     alpha=0.4)
ax.bar(x,         p_noisy,      width, label='Noisy',     color='tomato',    alpha=0.8)
ax.bar(x + width, p_corrected,  width, label='Corrected', color='steelblue', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(state_labels)
ax.set_xlabel("Measured state")
ax.set_ylabel("Probability")
ax.set_title("Measurement error mitigation on Bell state")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Zero-Noise Extrapolation

ZNE assumes $\langle O \rangle(\lambda) = \langle O \rangle_0 + a_1 \lambda + a_2 \lambda^2 + \cdots$
where $\lambda = 1$ is the natural noise level.

**Gate folding** amplifies noise by replacing $G$ with $G\,(G^\dagger G)^k$, giving $\lambda = 2k+1$.
**Circuit-level folding** applies this to the full unitary: replace $U$ with $U\,(U^\dagger U)^k$.

**Richardson extrapolation** (linear, scales 1 and 3):
$$\langle O \rangle_0 \approx \frac{3\,E_1 - E_3}{2}$$

**Richardson extrapolation** (quadratic, scales 1, 3, 5):
$$\langle O \rangle_0 \approx \frac{15\,E_1 - 10\,E_3 + 3\,E_5}{8}$$

In [ ]:
# ------------------------------------------------------------------ #
# Helper: <ZZ> from counts.
# ZZ eigenvalue for |ij> is (-1)^(i+j): +1 for even parity, -1 for odd.
# ------------------------------------------------------------------ #
def zz_expectation(counts, shots):
    ev = 0.0
    for bitstring, count in counts.items():
        parity = (-1) ** bitstring.count('1')
        ev += parity * count
    return ev / shots


# ------------------------------------------------------------------ #
# Circuit-level folding: append (U† U)^k to get noise scale 2k+1.
# qc must NOT contain measurements.
# ------------------------------------------------------------------ #
def fold_circuit(qc, scale):
    """Return circuit equivalent to U (U†U)^k where scale = 2k+1."""
    assert scale % 2 == 1 and scale >= 1, "scale must be a positive odd integer"
    k = (scale - 1) // 2
    folded = qc.copy()
    for _ in range(k):
        folded.compose(qc.inverse(), inplace=True)   # append U†
        folded.compose(qc, inplace=True)              # append U
    return folded


# Quick sanity check: fold at scale 3, verify the logical output is the same as scale 1
bell_base   = QuantumCircuit(2); bell_base.h(0); bell_base.cx(0, 1)
bell_folded = fold_circuit(bell_base, 3)
print(f"Bell circuit gate count:         {len(bell_base.data)}")
print(f"Folded (scale=3) gate count:     {len(bell_folded.data)}")
# Expected: 2 gates -> 6 gates (3x more, logically identical)

In [ ]:
# ------------------------------------------------------------------ #
# Noise model for ZNE demo: depolarizing on gate errors (no readout error)
# ------------------------------------------------------------------ #
p_1q = 0.005
p_2q = 0.040
nm_gate = NoiseModel()
nm_gate.add_all_qubit_quantum_error(depolarizing_error(p_1q, 1), ['sx', 'x', 'rz'])
nm_gate.add_all_qubit_quantum_error(depolarizing_error(p_2q, 2), ['cx'])
sim_gate = AerSimulator(noise_model=nm_gate)

shots_zne = 32768
scales    = [1, 3, 5]
evs       = []

for scale in scales:
    qc_folded = fold_circuit(bell_base, scale)
    qc_folded.measure_all()
    counts = sim_gate.run(transpile(qc_folded, sim_gate), shots=shots_zne).result().get_counts()
    ev = zz_expectation(counts, shots_zne)
    evs.append(ev)
    print(f"  scale {scale}: <ZZ> = {ev:.4f}")

# Richardson extrapolation
ev_linear = (3 * evs[0] - evs[1]) / 2
ev_quad   = (15 * evs[0] - 10 * evs[1] + 3 * evs[2]) / 8

print(f"\nIdeal  <ZZ>:                            1.0000")
print(f"Noisy  <ZZ> (scale = 1):               {evs[0]:.4f}")
print(f"ZNE linear   (scales 1, 3):            {ev_linear:.4f}")
print(f"ZNE quadratic (scales 1, 3, 5):        {ev_quad:.4f}")
# Expected: noisy ~0.93, ZNE linear ~0.99, ZNE quadratic ~0.997

In [ ]:
# Plot the measurements and both extrapolation curves
lambda_fine = np.linspace(-0.1, 5.5, 300)

# Linear fit through (1, evs[0]) and (3, evs[1])
b_lin = (evs[1] - evs[0]) / (3 - 1)
a_lin = evs[0] - b_lin

# Quadratic fit through all three points
coeffs    = np.polyfit([1, 3, 5], evs, 2)
poly_quad = np.poly1d(coeffs)

plt.figure(figsize=(7, 4))
plt.scatter(scales, evs, color='steelblue', zorder=5, s=60, label='Measured ⟨ZZ⟩')
plt.plot(lambda_fine, a_lin + b_lin * lambda_fine,
         color='tomato', linewidth=1.8, linestyle='--', label='Linear extrapolation')
plt.plot(lambda_fine, poly_quad(lambda_fine),
         color='seagreen', linewidth=1.8, linestyle=':', label='Quadratic extrapolation')
plt.scatter([0, 0], [ev_linear, ev_quad], marker='*', s=130,
            color=['tomato', 'seagreen'], zorder=5, label='Extrapolated at λ=0')
plt.axvline(0, color='gray', linewidth=0.8, linestyle=':')
plt.axhline(1.0, color='black', linewidth=0.8, linestyle=':', label='Ideal ⟨ZZ⟩ = 1')
plt.xlabel("Noise scale λ")
plt.ylabel("⟨ZZ⟩")
plt.title("Zero-noise extrapolation on Bell state")
plt.legend(loc='upper right')
plt.xlim(-0.4, 5.8)
plt.tight_layout()
plt.show()

In [ ]:
# Explore how ZNE performance degrades with circuit depth.
# Use repeated layers of H-CX as a proxy for depth.

def make_layered_circuit(n_layers):
    """n_layers repetitions of H(0)-CX(0,1). Logically equivalent to Bell for odd n_layers."""
    qc = QuantumCircuit(2)
    for _ in range(n_layers):
        qc.h(0)
        qc.cx(0, 1)
    return qc

layer_counts   = [1, 2, 4, 6, 8]
noisy_evs_deep = []
zne_evs_deep   = []

for n in layer_counts:
    # Ideal <ZZ> alternates: 1 for odd n, 0 for even n (H-CX is its own inverse)
    ideal = 1.0 if n % 2 == 1 else 0.0

    raw_evs = []
    for scale in [1, 3]:
        qc_f = fold_circuit(make_layered_circuit(n), scale)
        qc_f.measure_all()
        counts = sim_gate.run(transpile(qc_f, sim_gate), shots=shots_zne).result().get_counts()
        raw_evs.append(zz_expectation(counts, shots_zne))

    noisy_ev = raw_evs[0]
    zne_ev   = (3 * raw_evs[0] - raw_evs[1]) / 2
    noisy_evs_deep.append(abs(noisy_ev - ideal))
    zne_evs_deep.append(abs(zne_ev   - ideal))
    print(f"  {n} layers: ideal={ideal:.1f}, noisy={noisy_ev:.4f}, ZNE={zne_ev:.4f}")

plt.figure(figsize=(6, 3.5))
plt.plot(layer_counts, noisy_evs_deep, 'o-', color='tomato',    label='|noisy - ideal|')
plt.plot(layer_counts, zne_evs_deep,   's-', color='steelblue', label='|ZNE - ideal|')
plt.xlabel("Circuit depth (H-CX layer count)")
plt.ylabel("|error|")
plt.title("ZNE residual grows with circuit depth")
plt.legend()
plt.tight_layout()
plt.show()
# Expected: ZNE consistently reduces error but the gap shrinks at large depths.

## 3. Probabilistic Error Cancellation (Conceptual)

PEC expresses the inverse noise channel $\mathcal{E}_G^{-1}$ as a quasi-probability
distribution over implementable operations $\{B_i\}$:

$$\mathcal{E}_G^{-1} = \sum_i \alpha_i\, B_i \qquad \sum_i |\alpha_i| = \gamma_G \ge 1$$

By sampling operations from this distribution and averaging with signs, the noise is canceled
in expectation. The sampling overhead for a circuit with $d$ two-qubit gates is:

$$\text{samples} \sim \frac{\Gamma^2}{\epsilon^2} \qquad \Gamma = \prod_g \gamma_g$$

For 1% depolarizing error per gate, $\gamma_g \approx 1.04$.
At $d = 50$ gates: overhead $\approx 1.04^{50} \approx 7\times$.
At $d = 150$ gates with 3% error: overhead $> 10^6\times$.

PEC is available in Qiskit Runtime at `resilience_level = 3`.

In [ ]:
# PEC overhead estimate as a function of depth and error rate
depths      = np.arange(1, 101)
error_rates = [0.01, 0.02, 0.05]

plt.figure(figsize=(7, 4))
for p in error_rates:
    # For depolarizing noise: gamma_g = (1 + 3p/4) / (1 - 3p/4) approximately
    # A simpler first-order estimate: gamma_g ~ 1 + 2p for small p
    gamma_g  = 1 + 2 * p
    overhead = gamma_g ** (2 * depths)
    plt.semilogy(depths, overhead, label=f'p = {p*100:.0f}% per gate')

plt.axhline(100,  color='gray', linestyle=':', linewidth=0.8, label='100x overhead')
plt.axhline(1e6,  color='gray', linestyle='--', linewidth=0.8, label='10^6x overhead')
plt.xlabel("Number of two-qubit gates")
plt.ylabel("Sampling overhead Γ²")
plt.title("PEC sampling overhead vs circuit depth")
plt.legend()
plt.tight_layout()
plt.show()
# Expected: overhead grows exponentially; deeper circuits or higher error rates
# make PEC impractical very quickly.

## 4. Qiskit Runtime EstimatorV2

The `EstimatorV2` primitive automates transpilation, mitigation, and averaging.

**Resilience levels:**
| Level | Mitigation |
|-------|------------|
| 0 | None (raw noisy result) |
| 1 | Measurement error mitigation |
| 2 | ZNE + measurement error mitigation |
| 3 | PEC (expensive) |

**Required steps before calling `run`:**
1. Transpile the circuit to ISA format with `transpile(qc, backend)`.
2. Remap the observable: `obs.apply_layout(isa_circuit.layout)`.

Each entry in the PUB list is `(circuit, observable)` or `(circuit, observable, parameter_values)`.

In [ ]:
backend = FakeSherbrooke()

# Bell state circuit: ideal <ZZ> = 1.0
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

# Step 1: transpile to ISA (backend's native gate set and qubit layout)
isa_qc = transpile(qc, backend=backend, optimization_level=1, seed_transpiler=0)

# Step 2: remap observable from virtual to physical qubits
obs     = SparsePauliOp("ZZ")
isa_obs = obs.apply_layout(isa_qc.layout)

print(f"Virtual circuit depth:    {qc.depth()}")
print(f"Transpiled circuit depth: {isa_qc.depth()}")
print(f"Transpiled circuit gates: {isa_qc.count_ops()}")
# The transpiled circuit may be deeper than the original because H decomposes
# into native gates (sx, rz) and CX may map to ECR or another native gate.

In [ ]:
# Run at resilience levels 0, 1, and 2
level_labels = {0: 'No mitigation', 1: 'Readout mitigation', 2: 'ZNE'}
ev_results   = {}

for level, label in level_labels.items():
    options                = EstimatorOptions()
    options.resilience_level = level
    options.default_shots  = 8192

    est  = Estimator(mode=backend, options=options)
    job  = est.run([(isa_qc, isa_obs)])
    ev   = float(job.result()[0].data.evs)
    ev_results[label] = ev
    print(f"  Level {level} ({label:>20s}): <ZZ> = {ev:.4f}")

print(f"\n  Ideal: <ZZ> = 1.0000")
# Expected: level 0 ~0.89, level 1 ~0.92, level 2 ~0.98.
# Each level recovers more of the ideal value.

In [ ]:
# Retrieve the standard deviation from the ZNE run
options_zne                = EstimatorOptions()
options_zne.resilience_level = 2
options_zne.default_shots  = 8192

est_zne = Estimator(mode=backend, options=options_zne)
job_zne = est_zne.run([(isa_qc, isa_obs)])
data    = job_zne.result()[0].data

ev  = float(data.evs)
std = float(data.stds)

print(f"ZNE estimate: <ZZ> = {ev:.4f} ± {std:.4f}")
print(f"Ideal:        <ZZ> = 1.0000")
print()
print("The std captures both finite-shot uncertainty and extrapolation uncertainty.")
print("Increasing default_shots reduces the former; a better polynomial reduces the latter.")
# Expected: ev ~0.98, std ~0.01 to 0.02

In [ ]:
# Bar chart comparing all four scenarios
labels_plot = ['Ideal', 'No\nmitigation', 'Readout\nmitigation', 'ZNE\n(level 2)']
values      = [1.0, ev_results['No mitigation'], ev_results['Readout mitigation'], ev_results['ZNE']]
colors      = ['black', 'tomato', 'goldenrod', 'steelblue']

plt.figure(figsize=(7, 4))
bars = plt.bar(labels_plot, values, color=colors, alpha=0.8, edgecolor='white')
plt.axhline(1.0, color='black', linewidth=1, linestyle=':', label='Ideal = 1.0')
plt.ylabel("⟨ZZ⟩ expectation value")
plt.title("Mitigation levels on Bell state (FakeSherbrooke)")
plt.ylim(0.8, 1.05)
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 0.003,
             f"{val:.3f}", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 5. Exercises

Work through each exercise in the cell below it. Expected results are given so you can check.

### Exercise 1: Phase-flip readout mitigation

Build a 1-qubit noise model where:
- `P(read 1 | prepared 0) = 0.08` (p_ro_0)
- `P(read 0 | prepared 1) = 0.04` (p_ro_1)

This is asymmetric because the excited state $|1\rangle$ tends to decay to $|0\rangle$ during readout.

1. Build the 2×2 calibration matrix $M$ for this single-qubit noise model.
2. Prepare the state $|+\rangle = H|0\rangle$ and measure 8192 shots. Compute the noisy $P(0)$ and $P(1)$.
3. Apply $M^{-1}$ to correct the probabilities. What is the corrected $P(0)$?

*Expected: corrected $P(0) \approx 0.500$ since $|+\rangle$ has equal $Z$-basis probabilities.*

In [ ]:
# Exercise 1 — Asymmetric readout error

p_ro_0 = 0.08   # P(read 1 | prepared 0)
p_ro_1 = 0.04   # P(read 0 | prepared 1)

# TODO: build the ReadoutError and a 1-qubit AerSimulator
# ro_err_ex1 = ReadoutError([[1 - p_ro_0, p_ro_0], [p_ro_1, 1 - p_ro_1]])
# nm_ex1 = NoiseModel()
# nm_ex1.add_all_qubit_readout_error(ro_err_ex1)
# backend_ex1 = AerSimulator(noise_model=nm_ex1)

# TODO: calibrate the 2×2 matrix M
# (prepare |0> and |1>, measure each)

# TODO: prepare |+> and correct
# qc_plus = QuantumCircuit(1, 1)
# qc_plus.h(0)
# qc_plus.measure(0, 0)
# counts_plus = backend_ex1.run(transpile(qc_plus, backend_ex1), shots=8192).result().get_counts()
# p_noisy_1q = np.array([counts_plus.get('0', 0) / 8192, counts_plus.get('1', 0) / 8192])
# p_corrected_1q = np.linalg.inv(M_1q) @ p_noisy_1q
# p_corrected_1q = np.clip(p_corrected_1q, 0, 1) / p_corrected_1q.clip(0).sum()
# print(f"Corrected P(0): {p_corrected_1q[0]:.4f}")

### Exercise 2: ZNE on a deeper circuit

Build a GHZ state on 3 qubits: `H(0) - CX(0,1) - CX(0,2)`. The ideal $\langle ZZZ \rangle = 1$.

1. Implement a `zzz_expectation(counts, shots)` function: the $ZZZ$ eigenvalue for a 3-qubit bitstring
   is $(-1)^{\text{number of 1s}}$, same parity rule as for $ZZ$.
2. Apply circuit-level folding at scales 1 and 3.
3. Compute the linear Richardson estimate. By how much does ZNE improve the result?

*Expected: noisy $\langle ZZZ \rangle \approx 0.85$ to $0.90$, ZNE linear $\approx 0.95$ to $0.99$.*

In [ ]:
# Exercise 2 — ZNE on a 3-qubit GHZ state

def zzz_expectation(counts, shots):
    # TODO: same parity rule as zz_expectation but for 3 qubits
    pass


def make_ghz():
    # TODO: 3-qubit GHZ circuit (no measurements)
    qc = QuantumCircuit(3)
    # qc.h(0); qc.cx(0, 1); qc.cx(0, 2)
    return qc


# TODO: run at scales 1 and 3, compute Richardson linear estimate
# scales = [1, 3]
# for scale in scales:
#     qc_f = fold_circuit(make_ghz(), scale)
#     qc_f.measure_all()
#     counts = sim_gate.run(transpile(qc_f, sim_gate), shots=32768).result().get_counts()
#     ev = zzz_expectation(counts, 32768)
#     print(f"  scale {scale}: <ZZZ> = {ev:.4f}")

### Exercise 3: Runtime Estimator with multiple observables

`EstimatorV2` can evaluate several observables in a single PUB, or several PUBs in one `run` call.

1. Build a Bell state circuit and ISA-transpile it for FakeSherbrooke.
2. Create two observables: `SparsePauliOp("ZZ")` and `SparsePauliOp("XX")`.
   For the Bell state $|\Phi^+\rangle$, both have ideal expectation value 1.
3. Pass both as a single `SparsePauliOp.sum([obs1, obs2])` or as two separate PUBs in one `run` call.
4. Report $\langle ZZ \rangle$ and $\langle XX \rangle$ at resilience level 2.

*Expected: both should be close to 1 after ZNE, but XX may differ slightly because*
*transpilation introduces different gate sequences for the two observables.*

In [ ]:
# Exercise 3 — Multiple observables in one run call

# TODO:
# obs_ZZ = SparsePauliOp("ZZ")
# obs_XX = SparsePauliOp("XX")
# isa_obs_ZZ = obs_ZZ.apply_layout(isa_qc.layout)
# isa_obs_XX = obs_XX.apply_layout(isa_qc.layout)
#
# options = EstimatorOptions()
# options.resilience_level = 2
# options.default_shots = 8192
#
# est = Estimator(mode=backend, options=options)
# job = est.run([(isa_qc, isa_obs_ZZ), (isa_qc, isa_obs_XX)])
# results_multi = job.result()
# print(f"<ZZ> = {float(results_multi[0].data.evs):.4f}")
# print(f"<XX> = {float(results_multi[1].data.evs):.4f}")

## Summary

In this lesson you:

- Built a **calibration matrix** for 2-qubit readout errors by preparing each basis state and
  recording measurement outcomes. Applied $M^{-1}$ to recover corrected probability distributions,
  reducing the Bell state error from ~11% down to <1%.
- Implemented **circuit-level gate folding** (`fold_circuit`) to amplify noise at scales
  $\lambda = 1, 3, 5$ without changing the circuit's logical output. Applied **Richardson
  extrapolation** (linear and quadratic) to extrapolate back to $\lambda = 0$, recovering
  $\langle ZZ \rangle$ within 0.3% of the ideal value on a simple depolarizing noise model.
- Observed that ZNE residual error grows with circuit depth and that the quadratic
  estimate is more accurate at the cost of one additional circuit execution.
- Described **PEC** conceptually: it is unbiased but has exponential sampling overhead
  $\Gamma^2 \propto \gamma_g^{2d}$, making it practical only for shallow circuits.
- Used `EstimatorV2` from `qiskit_ibm_runtime` with `FakeSherbrooke` at resilience levels
  0, 1, and 2. Transpiled circuits to ISA format, remapped observables with `apply_layout`,
  and compared the mitigated expectation values and their standard deviations.

**Lesson 4** introduces quantum error **correction**: the three-qubit bit-flip code corrects
any single $X$ error deterministically, with no statistical overhead. The Shor code extends
this to arbitrary single-qubit errors. The stabilizer formalism provides the algebraic foundation
for all quantum error correcting codes, including the surface code.